### **Separate XGBoost per Product**

In [2]:
import pandas as pd
up_wide = pd.read_parquet("../data/petrochem/up_wide.parquet")

In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import plotly.graph_objects as go
from IPython.display import display

print("="*120)
print("APPROACH 1: SEPARATE XGBoost MODEL FOR EACH PRODUCT")
print("="*120)

# ============================================================================
# CELL 1: Data Preparation
# ============================================================================

print("\n[STEP 1] Preparing Data...")

product_tags = [
    'FIC10014_PV', 'FIC20017_PV', 'FIC20018_PV', 'FIC25021_PV', 
    'FIC30018_PV', 'FI40039_PV', 'FIC40045_PV', 'FIC50011_PV', 
    'FIC60020_PV', 'FIC65004_PV', 'FIC70053_PV', 'FIC95008_PV'
]

utility_tags = [tag for tag in up_wide.columns if tag not in product_tags]
product_tags = [t for t in product_tags if t in up_wide.columns]
utility_tags = [t for t in utility_tags if t in up_wide.columns]

X = up_wide[utility_tags].values
y = up_wide[product_tags].values

valid_idx = (~np.isnan(X).any(axis=1)) & (~np.isnan(y).any(axis=1))
X = X[valid_idx]
y = y[valid_idx]

scaler_X = StandardScaler()
X_norm = scaler_X.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42
)

print(f"  Utilities: {len(utility_tags)}, Products: {len(product_tags)}")
print(f"  Train: {X_train.shape[0]}, Test: {X_test.shape[0]} ✓")


APPROACH 1: SEPARATE XGBoost MODEL FOR EACH PRODUCT

[STEP 1] Preparing Data...
  Utilities: 20, Products: 12
  Train: 19686, Test: 4922 ✓


In [4]:
# ============================================================================
# CELL 2: Train Separate XGBoost for Each Product
# ============================================================================

print("\n[STEP 2] Training Separate XGBoost Models...")
print("="*120)

xgb_models = {}
xgb_metrics_list = []

for prod_idx, prod_tag in enumerate(product_tags):
    print(f"Training {prod_tag}...", end=" ")
    
    # Train individual model
    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=0
    )
    
    model.fit(X_train, y_train[:, prod_idx], verbose=False)
    
    # Predict
    y_pred_test = model.predict(X_test)
    y_test_prod = y_test[:, prod_idx]
    
    # Metrics
    r2 = r2_score(y_test_prod, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test_prod, y_pred_test))
    mae = mean_absolute_error(y_test_prod, y_pred_test)
    
    quality = '⭐ Excellent' if r2 > 0.85 else '✓ Good' if r2 > 0.75 else '⚠ Fair'
    
    xgb_models[prod_tag] = model
    xgb_metrics_list.append({
        'Product': prod_tag,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Quality': quality
    })
    
    print(f"R²: {r2:.4f} {quality}")

print("="*120)

xgb_metrics_df = pd.DataFrame(xgb_metrics_list)
#print(f"\nXGBoost Separate Models - Overall R²: {xgb_metrics_df['R²'].mean():.4f}")
display(xgb_metrics_df)




[STEP 2] Training Separate XGBoost Models...
Training FIC10014_PV... R²: 0.2619 ⚠ Fair
Training FIC20017_PV... R²: 0.6962 ⚠ Fair
Training FIC20018_PV... R²: 0.6027 ⚠ Fair
Training FIC25021_PV... R²: 0.7710 ✓ Good
Training FIC30018_PV... R²: 0.9441 ⭐ Excellent
Training FI40039_PV... R²: -0.0000 ⚠ Fair
Training FIC40045_PV... R²: 0.6705 ⚠ Fair
Training FIC50011_PV... R²: 0.9291 ⭐ Excellent
Training FIC60020_PV... R²: 0.9074 ⭐ Excellent
Training FIC65004_PV... R²: 0.8829 ⭐ Excellent
Training FIC70053_PV... R²: 0.9175 ⭐ Excellent
Training FIC95008_PV... R²: 0.8557 ⭐ Excellent


,Product,R²,RMSE,MAE,Quality
0,FIC10014_PV,0.261909,8.507790e+03,2.500322e+03,⚠ Fair
1,FIC20017_PV,0.696209,1.232014e+04,8.966920e+03,⚠ Fair
2,FIC20018_PV,0.602740,2.380717e+04,1.556065e+04,⚠ Fair
3,FIC25021_PV,0.771043,3.062650e+04,1.231980e+04,✓ Good
4,FIC30018_PV,0.944093,1.234342e+01,5.662550e+00,⭐ Excellent
5,FI40039_PV,-0.000048,1.621065e+32,1.348281e+31,⚠ Fair
6,FIC40045_PV,0.670534,1.929869e+03,1.415889e+03,⚠ Fair
7,FIC50011_PV,0.929077,3.180161e+03,2.139484e+03,⭐ Excellent
8,FIC60020_PV,0.907374,3.318121e+03,2.042361e+03,⭐ Excellent
9,FIC65004_PV,0.882914,1.022299e+03,7.207718e+02,⭐ Excellent


In [15]:
import pickle
import os

# Create folder
folder = 'saved_models'
os.makedirs(folder, exist_ok=True)

# Save models
for prod_tag, model in xgb_models.items():
    path = f'{folder}/{prod_tag}_xgboost_model.pkl'
    pickle.dump(model, open(path, 'wb'))

# Save scaler
pickle.dump(scaler_X, open(f'{folder}/scaler_X.pkl', 'wb'))

print(f"✓ All models saved to: {folder}/")

✓ All models saved to: saved_models/


In [ ]:
# ============================================================================
# CELL 3: Make Predictions & Calculate Efficiency
# ============================================================================

print("\n[STEP 3] Making Predictions on Full Dataset...")

y_pred_xgb = np.zeros_like(y)

for prod_idx, (prod_tag, model) in enumerate(xgb_models.items()):
    y_pred_xgb[:, prod_idx] = model.predict(X_norm)

efficiency_xgb = (y.sum(axis=1) / (y_pred_xgb.sum(axis=1) + 1e-10)) * 100

print(f"  Mean Efficiency: {efficiency_xgb.mean():.2f}%")
print(f"  Std Efficiency: {efficiency_xgb.std():.2f}%")



In [ ]:
# ============================================================================
# CELL 4: Visualize XGBoost Results
# ============================================================================

fig = go.Figure()

fig.add_trace(go.Bar(
    x=xgb_metrics_df['Product'],
    y=xgb_metrics_df['R²'],
    marker=dict(
        color=xgb_metrics_df['R²'],
        colorscale='RdYlGn',
        cmin=0,
        cmax=1,
        colorbar=dict(title='R² Score')
    ),
    text=np.round(xgb_metrics_df['R²'], 3),
    textposition='auto'
))

fig.update_layout(
    title='XGBoost (Separate Models) - R² per Product',
    xaxis_title='Product',
    yaxis_title='R² Score',
    height=500,
    width=1200,
    xaxis_tickangle=45
)

fig.show()

# Save
xgb_metrics_df.to_csv('xgb_separate_metrics.csv', index=False)
print("\n✓ Saved: xgb_separate_metrics.csv")

### **LightGBM**

In [5]:
import lightgbm as lgb

print("\n\n" + "="*120)
print("APPROACH 2: LightGBM (Separate Model per Product)")
print("="*120)

# ============================================================================
# CELL 5: Train LightGBM Models
# ============================================================================

print("\n[STEP 1] Training LightGBM Models...")
print("="*120)

lgb_models = {}
lgb_metrics_list = []

for prod_idx, prod_tag in enumerate(product_tags):
    print(f"Training {prod_tag}...", end=" ")
    
    # Create dataset for LightGBM
    train_data = lgb.Dataset(X_train, label=y_train[:, prod_idx])
    
    # Train
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1
    }
    
    model = lgb.train(params, train_data, num_boost_round=200)
    
    # Predict
    y_pred_test = model.predict(X_test)
    y_test_prod = y_test[:, prod_idx]
    
    # Metrics
    r2 = r2_score(y_test_prod, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test_prod, y_pred_test))
    mae = mean_absolute_error(y_test_prod, y_pred_test)
    
    quality = '⭐ Excellent' if r2 > 0.85 else '✓ Good' if r2 > 0.75 else '⚠ Fair'
    
    lgb_models[prod_tag] = model
    lgb_metrics_list.append({
        'Product': prod_tag,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Quality': quality
    })
    
    print(f"R²: {r2:.4f} {quality}")

print("="*120)

lgb_metrics_df = pd.DataFrame(lgb_metrics_list)
# print(f"\nLightGBM Separate Models - Overall R²: {lgb_metrics_df['R²'].mean():.4f}")
display(lgb_metrics_df)





APPROACH 2: LightGBM (Separate Model per Product)

[STEP 1] Training LightGBM Models...
Training FIC10014_PV... R²: 0.2644 ⚠ Fair
Training FIC20017_PV... R²: 0.6720 ⚠ Fair
Training FIC20018_PV... R²: 0.5786 ⚠ Fair
Training FIC25021_PV... R²: 0.7504 ✓ Good
Training FIC30018_PV... R²: 0.9413 ⭐ Excellent
Training FI40039_PV... R²: 0.0558 ⚠ Fair
Training FIC40045_PV... R²: 0.6715 ⚠ Fair
Training FIC50011_PV... R²: 0.9209 ⭐ Excellent
Training FIC60020_PV... R²: 0.8977 ⭐ Excellent
Training FIC65004_PV... R²: 0.8803 ⭐ Excellent
Training FIC70053_PV... R²: 0.9108 ⭐ Excellent
Training FIC95008_PV... R²: 0.8422 ✓ Good


,Product,R²,RMSE,MAE,Quality
0,FIC10014_PV,0.264410,8.493360e+03,2.635957e+03,⚠ Fair
1,FIC20017_PV,0.671967,1.280225e+04,9.377998e+03,⚠ Fair
2,FIC20018_PV,0.578586,2.452027e+04,1.627174e+04,⚠ Fair
3,FIC25021_PV,0.750386,3.197826e+04,1.334834e+04,✓ Good
4,FIC30018_PV,0.941282,1.264985e+01,5.906052e+00,⭐ Excellent
5,FI40039_PV,0.055764,1.575180e+32,2.007033e+31,⚠ Fair
6,FIC40045_PV,0.671534,1.926938e+03,1.414071e+03,⚠ Fair
7,FIC50011_PV,0.920860,3.359339e+03,2.244574e+03,⭐ Excellent
8,FIC60020_PV,0.897748,3.486267e+03,2.197148e+03,⭐ Excellent
9,FIC65004_PV,0.880321,1.033558e+03,7.325561e+02,⭐ Excellent


In [ ]:
# ============================================================================
# CELL 6: LightGBM Predictions & Efficiency
# ============================================================================

print("\n[STEP 2] Making Predictions...")

y_pred_lgb = np.zeros_like(y)

for prod_idx, (prod_tag, model) in enumerate(lgb_models.items()):
    y_pred_lgb[:, prod_idx] = model.predict(X_norm)

efficiency_lgb = (y.sum(axis=1) / (y_pred_lgb.sum(axis=1) + 1e-10)) * 100

print(f"  Mean Efficiency: {efficiency_lgb.mean():.2f}%")
print(f"  Std Efficiency: {efficiency_lgb.std():.2f}%")

# Save
#lgb_metrics_df.to_csv('lgb_separate_metrics.csv', index=False)
#print("\n✓ Saved: lgb_separate_metrics.csv")

### **CELL 7-11: Fixed Neural Network**

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print("\n\n" + "="*120)
print("APPROACH 3: FIXED NEURAL NETWORK (With Proper Scaling)")
print("="*120)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================================
# CELL 7: Data Prep with Proper Y Scaling
# ============================================================================

print("\n[STEP 1] Preparing Data with Proper Scaling...")

scaler_y = StandardScaler()
y_norm = scaler_y.fit_transform(y)

X_train_nn, X_test_nn, y_train_nn, y_test_nn = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42
)

class MIMODataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MIMODataset(X_train_nn, y_train_nn)
test_dataset = MIMODataset(X_test_nn, y_test_nn)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"  Data prepared with proper Y scaling ✓")



APPROACH 3: FIXED NEURAL NETWORK (With Proper Scaling)

[STEP 1] Preparing Data with Proper Scaling...
  Data prepared with proper Y scaling ✓


In [7]:
# ============================================================================
# CELL 8: Improved Neural Network Model
# ============================================================================

print("\n[STEP 2] Building Improved NN Model...")

class ImprovedMIMONetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super(ImprovedMIMONetwork, self).__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(0.1)
        
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(0.1)
        
        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.1)
        
        self.fc_out = nn.Linear(64, output_size)
        
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        
        x = self.fc_out(x)
        return x

model_nn = ImprovedMIMONetwork(len(utility_tags), len(product_tags)).to(device)

print(f"  Model architecture ready ✓")



[STEP 2] Building Improved NN Model...
  Model architecture ready ✓


In [8]:

# ============================================================================
# CELL 9: Train with Improved Settings
# ============================================================================

print("\n[STEP 3] Training Improved NN...")
print("="*120)

criterion = nn.HuberLoss(delta=1.0)  # Better than MSELoss
optimizer = optim.Adam(model_nn.parameters(), lr=0.0005, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

epochs = 150
nn_train_losses = []
nn_test_losses = []

for epoch in range(epochs):
    model_nn.train()
    train_loss = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        y_pred = model_nn(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model_nn.parameters(), 1.0)
        
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    nn_train_losses.append(train_loss)
    
    model_nn.eval()
    test_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model_nn(X_batch)
            loss = criterion(y_pred, y_batch)
            test_loss += loss.item()
    
    test_loss /= len(test_loader)
    nn_test_losses.append(test_loss)
    scheduler.step(test_loss)
    
    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs} | Train: {train_loss:.6f} | Test: {test_loss:.6f}")

print("="*120)
print("Training Complete! ✓")




[STEP 3] Training Improved NN...
Epoch  30/150 | Train: 0.108096 | Test: 0.089581
Epoch  60/150 | Train: 0.101128 | Test: 0.084615
Epoch  90/150 | Train: 0.100105 | Test: 0.083690
Epoch 120/150 | Train: 0.097978 | Test: 0.081131
Epoch 150/150 | Train: 0.095165 | Test: 0.079629
Training Complete! ✓


In [9]:
# ============================================================================
# CELL 10: NN Evaluation & Predictions
# ============================================================================

print("\n[STEP 4] Evaluating Improved NN...")

model_nn.eval()

with torch.no_grad():
    X_all_tensor = torch.FloatTensor(X_norm).to(device)
    y_pred_norm_nn = model_nn(X_all_tensor).cpu().numpy()

y_pred_nn = scaler_y.inverse_transform(y_pred_norm_nn)

nn_metrics_list = []

for prod_idx, prod_tag in enumerate(product_tags):
    r2 = r2_score(y[:, prod_idx], y_pred_nn[:, prod_idx])
    rmse = np.sqrt(mean_squared_error(y[:, prod_idx], y_pred_nn[:, prod_idx]))
    mae = mean_absolute_error(y[:, prod_idx], y_pred_nn[:, prod_idx])
    
    quality = '⭐ Excellent' if r2 > 0.85 else '✓ Good' if r2 > 0.75 else '⚠ Fair'
    
    nn_metrics_list.append({
        'Product': prod_tag,
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Quality': quality
    })

nn_metrics_df = pd.DataFrame(nn_metrics_list)
#print(f"\nImproved NN - Overall R²: {nn_metrics_df['R²'].mean():.4f}")
display(nn_metrics_df)



[STEP 4] Evaluating Improved NN...


,Product,R²,RMSE,MAE,Quality
0,FIC10014_PV,0.059376,8.851978e+03,2.267301e+03,⚠ Fair
1,FIC20017_PV,0.698840,1.275839e+04,8.910702e+03,⚠ Fair
2,FIC20018_PV,0.586942,2.506882e+04,1.478138e+04,⚠ Fair
3,FIC25021_PV,0.785474,3.066232e+04,1.212376e+04,✓ Good
4,FIC30018_PV,0.936715,1.349384e+01,6.401774e+00,⭐ Excellent
5,FI40039_PV,0.000600,1.503102e+32,7.119709e+30,⚠ Fair
6,FIC40045_PV,0.678601,1.936228e+03,1.423677e+03,⚠ Fair
7,FIC50011_PV,0.915445,3.526583e+03,2.316079e+03,⭐ Excellent
8,FIC60020_PV,0.893895,3.561756e+03,2.331541e+03,⭐ Excellent
9,FIC65004_PV,0.865826,1.100521e+03,7.932000e+02,⭐ Excellent


In [ ]:

efficiency_nn = (y.sum(axis=1) / (y_pred_nn.sum(axis=1) + 1e-10)) * 100

print(f"\nEnergy Efficiency:")
print(f"  Mean: {efficiency_nn.mean():.2f}%")
print(f"  Std: {efficiency_nn.std():.2f}%")

# nn_metrics_df.to_csv('nn_improved_metrics.csv', index=False)
# print("\n✓ Saved: nn_improved_metrics.csv")

### **CELL 12: Final Comparison**

In [14]:
print("\n\n" + "="*120)
print("FINAL COMPARISON: All 3 Approaches")
print("="*120)

comparison_df = pd.DataFrame({
    'Approach': ['XGBoost (Separate)', 'LightGBM (Separate)', 'Neural Network (Fixed)'],
    'Avg_R²': [
        xgb_metrics_df['R²'].mean(),
        lgb_metrics_df['R²'].mean(),
        nn_metrics_df['R²'].mean()
    ],
    'Excellent_Count': [
        (xgb_metrics_df['R²'] > 0.85).sum(),
        (lgb_metrics_df['R²'] > 0.85).sum(),
        (nn_metrics_df['R²'] > 0.85).sum()
    ],
    #'Mean_Efficiency_%': [
     #   efficiency_xgb.mean(),
      #  efficiency_lgb.mean(),
       # efficiency_nn.mean()
 #   ]
})

print("\n" + comparison_df.to_string(index=False))

# Find best approach
best_idx = comparison_df['Avg_R²'].idxmax()
best_approach = comparison_df.loc[best_idx, 'Approach']
best_r2 = comparison_df.loc[best_idx, 'Avg_R²']

print(f"\n" + "="*120)
print(f"🏆 WINNER: {best_approach} (R² = {best_r2:.4f})")
print("="*120)



FINAL COMPARISON: All 3 Approaches

              Approach   Avg_R²  Excellent_Count
    XGBoost (Separate) 0.703259                6
   LightGBM (Separate) 0.698822                5
Neural Network (Fixed) 0.683367                6

🏆 WINNER: XGBoost (Separate) (R² = 0.7033)
